## Setup: Load and audit the dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Synthetic data matching the course dataset
n = 4820
data = {
    'customer_id': [f'C-{i:04d}' for i in range(n)],
    'plan': np.random.choice(['starter', 'pro', 'enterprise', np.nan], n, p=[0.45, 0.35, 0.15, 0.05]),
    'support_tickets': np.where(np.random.random(n) < 0.14, np.nan, np.random.randint(0, 10, n)).astype(float),
    'monthly_spend': np.random.exponential(120, n),
    'churned': np.random.randint(0, 2, n),
}

df = pd.DataFrame(data)

print(f"Dataset shape: {df.shape}")
print(f"\nMissing data:")
print(df.isnull().sum())

## Bug 1: Missingness indicator created after imputation

The code below attempts to preserve the signal that `plan` missingness is predictive of churn. But there's an ordering issue.

In [ ]:
# --- BUGGY CODE ---
df['plan'] = df['plan'].fillna('unknown')
df['plan_was_missing'] = df['plan'].isnull().astype(int)

print(f"Rows with plan_was_missing=1: {df['plan_was_missing'].sum()}")
print(f"Expected: 437 (the original missing count), Got: {df['plan_was_missing'].sum()}")

**Explanation:** Write your answer here — why is `plan_was_missing` all zeros?

**Fix the bug:**

In [ ]:
# Reload the dataset to test the fix
df['plan'] = np.random.choice(['starter', 'pro', 'enterprise', np.nan], n, p=[0.45, 0.35, 0.15, 0.05])

# Correct order: indicator BEFORE imputation
df['plan_was_missing'] = df['plan'].isnull().astype(int)
df['plan'] = df['plan'].fillna('unknown')

print(f"Rows with plan_was_missing=1: {df['plan_was_missing'].sum()}")

## Bug 2: Mean imputation for a column where missing means zero

The `support_tickets` column is NaN when a customer has never opened a ticket — not when the data is unknown.

In [ ]:
# --- BUGGY CODE ---
df['support_tickets'] = df['support_tickets'].fillna(df['support_tickets'].mean())

print(f"Mean imputation value: {df['support_tickets'].mean():.2f}")
print(f"\nRows with imputed value near {df['support_tickets'].mean():.1f}: {(df['support_tickets'] > 3).sum()}")

**Explanation:** Write your answer here — what bias does mean imputation introduce when NaN really means zero?

**Fix the bug:**

In [ ]:
# Reload the dataset
df['support_tickets'] = np.where(np.random.random(n) < 0.14, np.nan, np.random.randint(0, 10, n)).astype(float)

# Correct fix: impute with zero since NaN means 'no tickets'
df['support_tickets'] = df['support_tickets'].fillna(0)

print(f"Min support_tickets: {df['support_tickets'].min()}")
print(f"Max support_tickets: {df['support_tickets'].max()}")
print(f"Rows with 0 tickets: {(df['support_tickets'] == 0).sum()}")

## Bug 3: Fitting a scaler on the full dataset before train/test split

The scaler learns statistics from test data, causing data leakage.

In [ ]:
# --- BUGGY CODE ---
X = df[['monthly_spend']].dropna()
y = df.loc[X.index, 'churned']

# Bug: fit_transform on FULL dataset before split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# NOW split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Scaler mean (learned from full dataset): {scaler.mean_[0]:.2f}")
print(f"Training set mean (before scaling): {X.iloc[:int(len(X)*0.8)].mean()[0]:.2f}")
print(f"Test set mean (before scaling): {X.iloc[int(len(X)*0.8):].mean()[0]:.2f}")

**Explanation:** Write your answer here — how does this contaminate the test evaluation?

**Fix the bug:**

In [ ]:
# Correct approach: split FIRST, then fit
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit scaler on training data ONLY
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaler mean (learned from training data): {scaler.mean_[0]:.2f}")
print(f"Training set mean (before scaling): {X_train.mean()[0]:.2f}")
print(f"These should match (scaler was fit on training data)")

## Summary check

Verify all three bugs have been fixed:

In [ ]:
# After all fixes, verify the state
print(f"Bug 1 - plan_was_missing preserved: {df['plan_was_missing'].sum()} rows (should be > 0)")
print(f"Bug 2 - support_tickets imputed with 0: min={df['support_tickets'].min()}, max={df['support_tickets'].max()}")
print(f"Bug 3 - scaler fit only on training data: mean from train={scaler.mean_[0]:.2f}")
print(f"\nAll bugs fixed: prepare for encoding in the next lesson.")